# BERT 기본개념

## Bidirectional Encoder Representations from Transformers

트랜스포머의 입력기만 떼어낸 부분이다.
> 트랜스포머의 인코더를 여러 개 쌓아 올린 구조임. 

특정 과제에 파인튜닝하면 NLP 최고 성능을 보인다!(몇십억 단어로 사전 훈련이 되어 있음)



## BERT 구조

1. BERT-Base  
L = 12 (인코더 층수. 트랜스포머에는 num_layer가 6이었다)  
D = 768 (임베딩 차원, d_model. 트랜스포머는 차원 수 보통 512였다.)  
A = 12 (셀프 어텐션헤드수. 트랜스포머는 8개로 나눠서 멀티헤드 어텐션 수행함)  
파라미터수: 1억1천만개(110M)
초기Transformer(L=6, D=512, A=8)보다 큰규모

2. BERT- Large 모델도 여기서 레이어 수, 다루는 임베딩 차원, 셀프 어텐션 헤드 수가 더 늘어난 것이고 그러니 당연히 w도 늘어난 것이다(BERT base 보다 3배 크다)

### BERT의 입력 구성

BERT가 한 번에 읽고 이해할 수 있는 입력 제한 사항 = 512 토큰과 두 문장
> BERT가 한 번에 처리할 수 있는 토큰이 512개이다. 즉, 문장 한 개든 두 개든 짧은 문장 10개이든 BERT는 한 번에 512 토큰만 받을 수 있다. 
> >입력 공식: [CLS] + 문장 A의 토큰들 + [SEP] + 문장 B의 토큰들 + [SEP] / 이때 여기 들어가는 모든 토큰의 총합이 512 이하여야 한다. 


> BERT는 단순히 문장 하나 이해를 넘어서 두 문장 사이의 관계를 파악하도록 설계되었는데, BERT의 주요 학습 목표 중 하나가 다음 문장 예측이기에 두 문장(두 개의 텍스트 덩어리)를 넣어줄 수 있다. (SEP 토큰을 기준으로 왼쪽 오른쪽)



BERT는 문맥을 완벽하게 파악하기 위해 단순히 단어만 넣는 것이 아니라, 세 가지 서로 다른 임베딩을 합쳐서(Sum) 하나의 완성된 입력을 만든다. 

BERT는 아래 세 가지 벡터를 더해서 최종 입력을 만듭니다. 
$Token + Segment + Position$


1. 토큰 임베딩: 단어 그 자체를 숫자로 바꾼 것(wordpiece 방식)


2. 세그먼트 임베딩: 이 토큰이 첫 번째 문장인지 두 번째 문장인지


3. 위치 임베딩

이 세개가 다 합쳐진다

**추가적으로 이 세가지 임베딩 과정은 특수토큰에도 적용된다**

특수토큰 CLS SEP도 결국 모델이 이해해야 하는 데이터이다.

CLS나 SEP 같은 토큰들도 토큰 임베딩, 세그먼트 임베딩, 위치임베딩과 더해져서 임베딩된다



-> 결국 BERT 입력 최종 모습(layer 들어가기 전에)은  행(토큰들)과 수백개의 열(임베딩 차원)로 이루어진 행렬이며
> 1행: cls 토큰(토큰 + 세그먼트 + 위치합산) / 2행 - N행: 실제 단어들 토큰(토큰 + 세그먼트 + 위치합산) / N+1행: sep 토큰(토큰 + 세그먼트 + 위치합산)일 것이다. 




## BERT의 학습 방식 두 가지 

BERT에는 사전학습 방식이 두 가지 있는데,
1. MLM
2. NSP 

가 그것이다.

이 학습 방식은 각각 토큰 단위에서, 시퀀스 단위에서 문맥 잘 이해하도록 사전 학습 시키는 방식이다. 그러므로 BERT 모델은 단어들(토큰)간의 문맥 뿐 아니라 문장 간의 문맥도 잘 학습한다. (BERT가 문맥 잘 이해하는 이유)

추가로 두 학습 방식 모두 **자기지도학습**이다. 

### MLM
*자기지도학습*  

토큰 단위에서 문맥을 잘 이해하나?

입력 단어를 무작위로 선택하여 가려진 단어를 예측하도록 학습 




Maksed Language Model 학습 방식이 왜 유효할까?
> 각각 토큰들은 서로 쌍대비교로 문맥 이해했는지 확인하려면 하나 가려두고 여기 토큰들 정도는 맞춰야지! 이렇게 학습시키는 것이다. 

> 우리가 아는 딥러닝 학습과는 좀 다르다. 딥러닝은 마지막 출력 가지고 틀렸어 다시~ 이거였는데 (loss 계산하고) BERT는 가리고 맞춘다(MLM) 



딥러닝은 지도학습이다. 지도학습에서는 문장과 정답을 같이 만들어준다.
근데 BERT는 정답 셋트가 필요없다. 정답 셋트 만들어주지 않아도 된다. -> 자기지도학습이다!!(self supervised learning) -> 이게 훨씬 쉬운 방식이다. 


밤새서 학습 데이터 만들 필요가 없다(딥러닝 위한 학습데이터)
그냥 문서만 가져오면 다 되는 것이다!
-> 그러니까 수천 수만개 학습이 가능하다
-> 자기지도학습이 가능하기 떄문에 엄청난 양의 데이터 가능함(지구상의 모든 문서) 


## NSP
*자기지도학습*

시퀀스 단위에서 문맥 잘 이해하나? (즉, 토큰 단위가 아니다. 이어지는 문장인지 관련 없는 문장인지 시퀀스 단위에서 문맥 충분히 이해하도록 한다. )

두개의 문장을 주고 두 번째 문장이 이어지는 문장인지 맞추는 학습.
실제 이어지는 문장쌍50% + 무작위로 이어붙인 문장 쌍 50%로 학습
> 즉, 이진분류임. 두 문장을 입력받고 ISNext NotNext 중에 하나 선택하고 맞추는 학습 과정에서 BERT 모델은 문장 간의 관계(앞 뒤 문장)가 어울리는지, 문맥을 통합하는 능력을 기르게 된다. 
>
> 실제 이어지는 문장인지 아니었는지 마지막 결과를 정답과 비교하며 가중치를 수정하고 그 과정으로 학습한다. (역전파)

## Fin - tunning 
### 사전학습된 BERT를 맞춤형으로 만들자!

**BERT 자체(Pre-training)는 자기지도학습(Self-Supervised Learning)**이 맞지만, 우리가 특정 데이터를 가져와서 수행하는 **파인튜닝(Fine-tuning)은 전형적인 지도학습(Supervised Learning)**이다. 

1. 사전 학습(Pre-training): 자기지도학습
BERT가 처음 태어날 때 거치는 과정

데이터: 정답(Label)이 없는 방대한 양의 일반 텍스트(위키피디아 등).

방법: 문장에서 단어 일부를 가리고(Masking) 원래 뭐가 있었는지 맞히게 한다.  + 다음 문장을 예측하게 한다. (NSP)

특징: 사람이 일일이 "이 문장은 긍정이야"라고 정답을 달아주지 않아도, 텍스트 그 자체를 정답으로 삼아 스스로 학습한다. 그래서 **'자기(Self)'가 스스로를 '지도(Supervised)'**한다고 부른다.

목적: 언어의 전반적인 맥락과 문법을 파악하는 '기초 체력' 기르기.


2. 파인튜닝(Fine-tuning): 지도학습
기초 체력을 기른 BERT를 데려와서 우리가 원하는 특정 작업(예: 감성 분석)을 시키는 과정이다. 

데이터: 사람이 직접 정답을 달아놓은 데이터셋 (예: "영화가 재밌다" = 긍정, "지루하다" = 부정).

방법: 모델에게 문장을 보여주고 우리가 정해준 정답(Label)을 맞히게 한다. 틀리면 가중치를 수정!

특징: **명확한 정답(Label)**이 존재하며, 이를 통해 모델을 특정 목적에 맞게 교정합니다. 이것이 바로 전형적인 지도학습의 모습이다. 

목적: 특정 도메인(사회과학 텍스트, 법률 문서 등)에서 문제를 해결하는 '전문화된 능력' 기르기.

3. 왜 이렇게 두 단계를 나누나요? 

사전 학습: 수조 개의 문장을 읽어 "한국어라는 언어의 사회적 맥락"을 통달한 언어학자를 만드는 과정입니다. (자기지도학습)

파인튜닝: 이 언어학자에게 "혐오 표현 1,000개"를 보여주며 어떤 게 혐오인지 가르쳐서 "혐오 표현 탐지기"로 만드는 과정입니다. (지도학습)

만약 처음부터 혐오 표현만 지도학습으로 가르쳤다면, 모델은 언어의 깊은 맥락을 몰라서 성능이 떨어졌을 것이다.. 하지만 자기지도학습으로 쌓은 기초 위에 지도학습을 얹었기 때문에 적은 데이터로도 엄청난 성능을 내는 것이다!! 그래서 RNN 보다 성능이 훨씬 좋다. 
> RNN은 아무 학습 안되어 있는 친구를 처음부터 학습시키는 거라면, 파인튜닝은 문맥 이해 능력을 가지고 있는 애한테 학습시키는 것이니까 아주 조그마한 데이터셋으로 파인튜닝해도 성능이 좋다!

## BERT는 결국..

결국 BERT는 토큰들에 대한 상호 이해! 그거만 뱉어내는 것에서 끝나는 것이다 (트랜스포머에서 디코더는 없으니까 문장 생성은 아니지)

-> 그래서 이걸 가지고 문장 분류 같은 것들을 하려고 하는 것이다. 그걸 위해서 파인튜닝을 해서 우리가 원하는 일을 하는 것이다!

BERT의 layer에서 트랜스포머의 인코더와 동일한 연산이 일어난다(12layer 니까 12번 일어남) 이 layer에 들어가기 전에 데이터는 Mask 토큰을 골라 가짜 토큰으로 갈아끼우고(MLM 준비), 두 문장이 이어지는 문장인지 상관없는 문장인지 미리 라벨링(Isnext=1, NotNext=0이런식으로)을 해둔다(NSP 준비)

이 과정에서 mask 토큰 주변의 단어들이 어텐션 연산을 하며 주변 맥락을 수집하고(MLM) 또 cls 토큰은 문장 전체 정보를 어텐션으로 모아 이 두 문장은 서로 연관된 것인가..?를 판단할 근거를 학습한다

12번 레이어를 통과하고 나오면 각 토큰 위치마다 768차원 벡터들이 나오고, 이때 MLM과 NSP가 동시에 수행된다. 
> mask 자리에 있던 출력 벡터를 꺼내서 예측한다. 또 CLS 토큰 자리에 있는 출력 벡터를 꺼내서 IsNext인가 NotNext인가 예측한다. (CLS는 문장 전체의 대변인이기 때문에 마지막 층에서 CLS 토큰에는 A 문장과 B문장이 전체적으로 어떤 분위기 인지 추측하는 문맥을 반영한다.)

결국 모델이 이 두가지를 맞추려고 학습하다 보면 12 layer의 가중치(w_q, w_k, w_v 등)이 언어의 맥락을 파악하도록 조정되는 것이다. 